# 10 — Recommender

Whether the recommendation layer (`recommender.py`) finds a student's weakest topics, checked with simulated students whose true ability per topic is known.

**Running it.** Every section below is the evaluation's own code. With `RUN = False`
(the default) nothing is recomputed: the results saved in `data/eval/` are loaded
and shown. Set `RUN = True` in the first code cell to measure again, which
overwrites those files. The simulation needs no models and runs quickly.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## Simulated students

Simulation study of the recommendation layer (recommender.py).

Real students' abilities are unknown, so the standard way to check an
adaptive tutor is with simulated students whose true ability per topic is
known. Each simulated student answers questions generated by a DIFFERENT
response model from the one the recommender assumes, so the recommender is
not simply being tested against itself:

```text
    P(correct) = c + (1 - c - slip) * sigmoid(a * (true_ability - b))
```

with discrimination a = 1.7, a 5% slip rate, and a 25% guessing floor c on
multiple-choice (level 1) questions. The recommender assumes a = 1, no slips
and no guessing.

Four question-selection policies are compared. All four feed the same
estimator (Progress), so only the choice of question differs:

```text
  adaptive      — topic = recommend()[0], level = next_level()   (the app)
  explore_first — each topic once (in order) first, then as adaptive
  round_robin   — topics in turn, always level 2
  random        — random topic, random level
```

Measured after a budget of N questions, averaged over simulated students:

```text
  weakest_top1 / weakest_top2 — the truly weakest topic is ranked first /
                                in the first two recommendations
  spearman                    — rank correlation between estimated mastery
                                and true ability over all topics
  practice_on_weakest_two     — share of questions spent on the two truly
                                weakest topics
  success_gap                 — mean |true P(correct) - 0.7| of the questions
                                asked; lower = difficulty better matched
```

In [3]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook']

In [4]:
import json
import math
import os
import random
import sys
from pathlib import Path
from statistics import mean

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from backend.pipeline.recommender import (  # noqa: E402
    LEVEL_DIFFICULTY, TARGET_SUCCESS, Progress)

OUT_PATH = ROOT / "data" / "eval" / "recommender_simulation.json"

N_STUDENTS = 500
N_TOPICS = 6
BUDGETS = [6, 12, 24, 36]
DISCRIMINATION = 1.7
SLIP = 0.05
GUESS = {1: 0.25, 2: 0.0, 3: 0.0}
SEED = 2026

In [5]:
def true_p(ability, level):
    s = 1.0 / (1.0 + math.exp(-DISCRIMINATION * (ability - LEVEL_DIFFICULTY[level])))
    return GUESS[level] + (1 - GUESS[level] - SLIP) * s

In [6]:
def rank(values):
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2
        i = j + 1
    return ranks

In [7]:
def spearman(x, y):
    rx, ry = rank(x), rank(y)
    mx, my = mean(rx), mean(ry)
    cov = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    var = math.sqrt(sum((a - mx) ** 2 for a in rx) * sum((b - my) ** 2 for b in ry))
    return cov / var if var else 0.0

In [8]:
def run_student(policy, abilities, budget, rng):
    topics = [f"T{i}" for i in range(len(abilities))]
    progress = Progress()
    weakest_two = set(sorted(topics, key=lambda t: abilities[t])[:2])
    on_weak, gaps = 0, []

    for step in range(budget):
        if policy == "explore_first" and step < len(topics):
            topic = topics[step]
            level = progress.next_level(topic)
        elif policy in ("adaptive", "explore_first"):
            topic = progress.recommend(topics, n=1)[0].topic_id
            level = progress.next_level(topic)
        elif policy == "round_robin":
            topic, level = topics[step % len(topics)], 2
        else:
            topic, level = rng.choice(topics), rng.choice(list(LEVEL_DIFFICULTY))

        p = true_p(abilities[topic], level)
        gaps.append(abs(p - TARGET_SUCCESS))
        on_weak += topic in weakest_two
        progress.record(topic, level, rng.random() < p)

    ranked = [r.topic_id for r in progress.recommend(topics, n=len(topics))]
    weakest = min(topics, key=lambda t: abilities[t])
    return {
        "weakest_top1": ranked[0] == weakest,
        "weakest_top2": weakest in ranked[:2],
        "spearman": spearman([progress.mastery(t) for t in topics],
                             [abilities[t] for t in topics]),
        "practice_on_weakest_two": on_weak / budget,
        "success_gap": mean(gaps),
    }

In [9]:
def main():
    rng = random.Random(SEED)
    students = [{f"T{i}": rng.gauss(0.0, 1.0) for i in range(N_TOPICS)}
                for _ in range(N_STUDENTS)]

    results = {}
    for budget in BUDGETS:
        results[budget] = {}
        for policy in ("adaptive", "explore_first", "round_robin", "random"):
            policy_rng = random.Random(SEED + budget)
            runs = [run_student(policy, s, budget, policy_rng) for s in students]
            results[budget][policy] = {
                key: round(mean(float(r[key]) for r in runs), 3) for key in runs[0]}

    chance_top1, chance_top2 = 1 / N_TOPICS, 2 / N_TOPICS
    print(f"{N_STUDENTS} simulated students, {N_TOPICS} topics "
          f"(chance: top-1 {chance_top1:.3f}, top-2 {chance_top2:.3f}, "
          f"weakest-two share {2 / N_TOPICS:.3f})\n")
    header = f"{'N':>3} {'policy':<14} {'top1':>6} {'top2':>6} {'rho':>6} {'weak2':>6} {'gap':>6}"
    print(header)
    for budget, by_policy in results.items():
        for policy, m in by_policy.items():
            print(f"{budget:>3} {policy:<14} {m['weakest_top1']:>6} {m['weakest_top2']:>6} "
                  f"{m['spearman']:>6} {m['practice_on_weakest_two']:>6} "
                  f"{m['success_gap']:>6}")
        print()

    OUT_PATH.write_text(json.dumps({
        "note": "Simulated students; response model differs from the "
                "recommender's (see script docstring).",
        "students": N_STUDENTS, "topics": N_TOPICS, "seed": SEED,
        "response_model": {"discrimination": DISCRIMINATION, "slip": SLIP,
                           "guess_by_level": GUESS,
                           "level_difficulty": LEVEL_DIFFICULTY},
        "chance": {"weakest_top1": chance_top1, "weakest_top2": chance_top2,
                   "practice_on_weakest_two": 2 / N_TOPICS},
        "results_by_budget": results,
    }, indent=2) + "\n", encoding="utf-8")
    print(f"  -> {OUT_PATH.relative_to(ROOT)}")

In [10]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [11]:
rs = saved("recommender_simulation.json")
print(f"{rs['students']} students, {rs['topics']} topics; chance: {rs['chance']}")
rows = []
for budget, policies in rs["results_by_budget"].items():
    for policy, r in policies.items():
        rows.append({"questions answered": int(budget), "policy": policy, **r})
pd.DataFrame(rows).set_index(["questions answered", "policy"]).round(3)

500 students, 6 topics; chance: {'weakest_top1': 0.16666666666666666, 'weakest_top2': 0.3333333333333333, 'practice_on_weakest_two': 0.3333333333333333}


weakest_top1  weakest_top2  spearman  \
questions answered policy                                                
6                  adaptive              0.334         0.474     0.197   
                   explore_first         0.326         0.510     0.280   
                   round_robin           0.292         0.526     0.478   
                   random                0.310         0.542     0.361   
12                 adaptive              0.392         0.536     0.284   
                   explore_first         0.412         0.566     0.330   
                   round_robin           0.380         0.680     0.601   
                   random                0.374         0.634     0.483   
24                 adaptive              0.498         0.606     0.388   
                   explore_first         0.540         0.640     0.454   
                   round_robin           0.516         0.798     0.714   
                   random                0.508         0.740     0.611   
36                 adaptive              0.578         0.666     0.441   
                   explore_first         0.580         0.688     0.480   
                   round_robin           0.592         0.836     0.750   
                   random                0.564         0.798     0.688   

                                  practice_on_weakest_two  success_gap  
questions answered policy                                               
6                  adaptive                         0.451        0.161  
                   explore_first                    0.333        0.163  
                   round_robin                      0.333        0.286  
                   random                           0.339        0.313  
12                 adaptive                         0.510        0.165  
                   explore_first                    0.466        0.165  
                   round_robin                      0.333        0.286  
                   random                           0.333        0.308  
24                 adaptive                         0.614        0.166  
                   explore_first                    0.600        0.167  
                   round_robin                      0.333        0.286  
                   random                           0.330        0.312  
36                 adaptive                         0.689        0.167  
                   explore_first                    0.678        0.169  
                   round_robin                      0.333        0.286  
                   random                           0.340        0.310